# Data Transformation

Data Transformation is the process of converting raw data into a format that is suitable for Machine Learning algorithms.

Many ML models cannot work directly with categorical values or features having different scales. Therefore, encoding, scaling, and feature engineering are performed before training the model.

In this notebook, we will learn different techniques used to transform data into a machine-learning-ready format.

In [1]:
## Import Libraries

import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import (LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures) 


In [2]:
## Load Dataset 

df = sns.load_dataset("titanic")

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# Encoding Categorical Variables

Machine Learning algorithms cannot understand text values directly.

Categorical variables must be converted into numerical values before training a model.

There are different encoding techniques depending on the type of categorical feature.

## One-Hot Encoding

One-Hot Encoding creates a separate binary column for every category.

It is commonly used for nominal categorical variables where there is no natural order.

In [5]:
encoded_data = pd.get_dummies(df, columns= ["embarked"])

encoded_data.head()

,survived,pclass,sex,age,sibsp,parch,fare,class,who,adult_male,deck,embark_town,alive,alone,embarked_C,embarked_Q,embarked_S
0,0,3,male,22.0,1,0,7.2500,Third,man,True,NaN,Southampton,no,False,False,False,True
1,1,1,female,38.0,1,0,71.2833,First,woman,False,C,Cherbourg,yes,False,True,False,False
2,1,3,female,26.0,0,0,7.9250,Third,woman,False,NaN,Southampton,yes,True,False,False,True
3,1,1,female,35.0,1,0,53.1000,First,woman,False,C,Southampton,yes,False,False,False,True
4,0,3,male,35.0,0,0,8.0500,Third,man,True,NaN,Southampton,no,True,False,False,True


## Label Encoding

Label Encoding assigns an integer value to each category.

It is mostly used for binary variables or ordinal features.

In [7]:
label = LabelEncoder()

df["sex_encoded"] = label.fit_transform(df["sex"])

df[["sex", "sex_encoded"]].head()

,sex,sex_encoded
0,male,1
1,female,0
2,female,0
3,female,0
4,male,1


## Target Encoding

Each category is replaced with the mean value of the target variable.

This technique is useful for high-cardinality categorical features.

In [8]:
target_encoding = (df.groupby("embarked")["survived"].mean())

df["embarked_target"] = df["embarked"].map(target_encoding)

df[["embarked","embarked_target"]].head()

,embarked,embarked_target
0,S,0.336957
1,C,0.553571
2,S,0.336957
3,S,0.336957
4,S,0.336957


## Frequency Encoding

Each category is replaced by its frequency in the dataset.

It reduces dimensionality while preserving category occurrence information.

In [9]:
freq = df["embarked"].value_counts()

df["embarked_frequency"] = df["embarked"].map(freq)

df[["embarked","embarked_frequency"]].head()

,embarked,embarked_frequency
0,S,644.0
1,C,168.0
2,S,644.0
3,S,644.0
4,S,644.0


# Numerical Feature Transformation

Numerical features often have different scales.

Scaling ensures that all features contribute equally during model training.

## Standard Scaling

StandardScaler transforms the data so that:

- Mean = 0
- Standard Deviation = 1

It works well when the data follows a normal distribution.

In [10]:
scaler = StandardScaler()

df["age_standard"] = scaler.fit_transform(df[["age"]])

df[["age","age_standard"]].head()

,age,age_standard
0,22.0,-0.530377
1,38.0,0.571831
2,26.0,-0.254825
3,35.0,0.365167
4,35.0,0.365167


## Min-Max Scaling

This method scales values between 0 and 1.

It is commonly used in Neural Networks and distance-based algorithms.

In [11]:
min_max = MinMaxScaler()

df["fare_min_max"] = min_max.fit_transform(df[["fare"]])

df[["fare","fare_min_max"]].head()

,fare,fare_min_max
0,7.2500,0.014151
1,71.2833,0.139136
2,7.9250,0.015469
3,53.1000,0.103644
4,8.0500,0.015713


## Robust Scaling

RobustScaler uses the median and interquartile range (IQR).

It performs well when the dataset contains outliers.

In [12]:
robust = RobustScaler()

df["fare_robust"] = robust.fit_transform(df[["fare"]])

df[["fare","fare_robust"]].head()

,fare,fare_robust
0,7.2500,-0.312011
1,71.2833,2.461242
2,7.9250,-0.282777
3,53.1000,1.673732
4,8.0500,-0.277363


## Log Transformation

Log Transformation reduces skewness and minimizes the impact of extremely large values.

In [13]:
df["fare_log"] = np.log1p(df["fare"])

df[["fare", "fare_log"]].head()

,fare,fare_log
0,7.2500,2.110213
1,71.2833,4.280593
2,7.9250,2.188856
3,53.1000,3.990834
4,8.0500,2.202765


## Square Root Transformation

Square Root Transformation is another technique used to reduce skewness in numerical data.

In [14]:
df["age_sqrt"] = np.sqrt(df["age"])

df[["age", "age_sqrt"]].head()

,age,age_sqrt
0,22.0,4.690416
1,38.0,6.164414
2,26.0,5.099020
3,35.0,5.916080
4,35.0,5.916080


## Polynomial Features

Polynomial Features create new features by combining existing numerical variables.

They help models capture non-linear relationships.

In [20]:
poly = PolynomialFeatures(degree=2, include_bias=False)

poly_features = poly.fit_transform(df[["age", "fare"]].fillna(0))

pd.DataFrame(poly_features, columns=poly.get_feature_names_out(["age","fare"])).head()


,age,fare,age^2,age fare,fare^2
0,22.0,7.2500,484.0,159.5000,52.562500
1,38.0,71.2833,1444.0,2708.7654,5081.308859
2,26.0,7.9250,676.0,206.0500,62.805625
3,35.0,53.1000,1225.0,1858.5000,2819.610000
4,35.0,8.0500,1225.0,281.7500,64.802500


# Feature Engineering

Feature Engineering creates new variables from existing data.

Well-designed features often improve model performance more than changing the algorithm itself.

## Polynomial Features

In [29]:
df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())

In [30]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)

df_poly = poly.fit_transform(df[["age", "fare"]])

print(df_poly[:5])

[[1.00000000e+00 2.20000000e+01 7.25000000e+00 4.84000000e+02
  1.59500000e+02 5.25625000e+01]
 [1.00000000e+00 3.80000000e+01 7.12833000e+01 1.44400000e+03
  2.70876540e+03 5.08130886e+03]
 [1.00000000e+00 2.60000000e+01 7.92500000e+00 6.76000000e+02
  2.06050000e+02 6.28056250e+01]
 [1.00000000e+00 3.50000000e+01 5.31000000e+01 1.22500000e+03
  1.85850000e+03 2.81961000e+03]
 [1.00000000e+00 3.50000000e+01 8.05000000e+00 1.22500000e+03
  2.81750000e+02 6.48025000e+01]]


## Binning

In [31]:
bins = [0,12,18,35,60,np.inf]

labels = ["Child", "Teen", "Young Adult", "Adult", "Senior"]

df["age_group"] = pd.cut(df["age"], bins = bins, labels = labels)

df[["age", "age_group"]].head()

,age,age_group
0,22.0,Young Adult
1,38.0,Adult
2,26.0,Young Adult
3,35.0,Young Adult
4,35.0,Young Adult


## Interaction Features

In [32]:
df["age_fare"] = (df["age"]*df["fare"])

df[["age","fare","age_fare"]].head()

,age,fare,age_fare
0,22.0,7.2500,159.5000
1,38.0,71.2833,2708.7654
2,26.0,7.9250,206.0500
3,35.0,53.1000,1858.5000
4,35.0,8.0500,281.7500


## Date Feature Extraction

In [ ]:
dates = pd.DataFrame({"date" : pd.date_range(starts="2024-01-01",periods=5)})

dates["years"] = dates["date"].dt.year
dates["month"] = dates["date"].dt.month
dates["day"] = dates["date"].dt.day

dates

- Note: The Titanic dataset does not contain a date column, so a sample dataset is used to demonstrate date feature extraction.

## Aggregation

In [ ]:
df["age_fare_sum"] = (df["age"] + df["fare"])

df[["age","fare","age_fare_sum"]].head()

---
## **📝 Note**

Feature transformation and engineering help convert raw data into meaningful inputs that improve model learning and prediction accuracy.

Choosing the right transformation techniques depends on the dataset, feature types, and the Machine Learning algorithm being used.